<a href="https://colab.research.google.com/github/sky-blue/manim/blob/main/manim_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!sudo apt update
!sudo apt install -y libcairo2-dev libpango1.0-dev ffmpeg texlive-latex-base texlive-fonts-recommended texlive-latex-extra
!pip install manim

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Fetched 3,917 B in 1s (2,851 B/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
112 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illin

In [2]:
%%writefile scene.py
from manim import *

class NormalizeBinary(Scene):
    def construct(self):
        self.camera.background_color = "#101526"
        print("STEP 1: Scene started")
        # Title
        title = Text(
            "Normalization",
            font_size=36,
            color=WHITE
        ).to_edge(UP).shift(DOWN * 0.25)

        self.play(Write(title))
        print("STEP 2: Title completed")
        self.wait(0.5)


        # 1. Initial binary representation

        dec_label = MathTex(
            r"0.1_{10} =",
            font_size=44
        )

        # Long binary expansion
        fraction_bits = "0001" + "1001" * 20

        digits = ["0", "."] + list(fraction_bits) + [r"\ldots"]

        all_mobs = VGroup(
            *[MathTex(d, font_size=44) for d in digits]
        )

        digit_mobs = VGroup(*all_mobs[:-1])
        ellipsis = all_mobs[-1]

        # Create multiple rows
        binary_rows = VGroup()
        row_width = 20

        for i in range(0, len(digit_mobs), row_width):
            row = VGroup(
                *digit_mobs[i:i + row_width]
            ).arrange(RIGHT, buff=0.08)

            binary_rows.add(row)

        # Add ellipsis to the last row
        binary_rows[-1].add(ellipsis)
        binary_rows[-1].arrange(RIGHT, buff=0.08)

        # Left-align all rows
        binary_rows.arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.08
        )

        # Align the top-left of the binary area with the decimal label
        start_eq = VGroup(
            dec_label,
            binary_rows
        ).arrange(
            RIGHT,
            buff=0.2,
            aligned_edge=UP
        )

        start_eq.move_to(UP * 0.3)

        # Lower the decimal point slightly
        digit_mobs[1].shift(DOWN * 0.13)

        self.play(
            Write(dec_label),
            LaggedStart(
                *[FadeIn(mob) for mob in digit_mobs],
                FadeIn(ellipsis),
                lag_ratio=0.003
            ),
            run_time=1
        )

        self.wait(1.5)


        # Keep 12 fractional digits and erase the rest
        bin_mobs = VGroup(*digit_mobs[:14])
        tail_digits = list(digit_mobs[14:])

        # Arrange the remaining binary digits in one line
        target_bin = VGroup(
            *[mob.copy() for mob in bin_mobs]
        ).arrange(RIGHT, buff=0.08)

        # Fix the decimal point position
        target_bin[1].shift(DOWN * 0.13)

        # Add ellipsis after the remaining digits
        target_ellipsis = MathTex(
            r"\ldots",
            font_size=44
        ).next_to(target_bin, RIGHT, buff=0.08)

        # Create the complete equation
        target_dec = dec_label.copy()

        target_equation = VGroup(
            target_dec,
            target_bin,
            target_ellipsis
        ).arrange(RIGHT, buff=0.2)

        # Move the complete equation to the screen center
        target_equation.move_to(ORIGIN)

        # Erase the remaining digits and move the equation
        self.play(
            LaggedStart(
                *[
                    FadeOut(char)
                    for char in reversed(tail_digits)
                ],
                lag_ratio=0.008
            ),
            FadeOut(ellipsis),
            Transform(bin_mobs, target_bin),
            Transform(dec_label, target_dec),
            FadeIn(target_ellipsis),
            run_time=3
        )

        self.wait(0.5)

        # 2. Highlight the first 1
        first_one = bin_mobs[5]

        self.play(
            first_one.animate.set_color(GREEN)
        )
        self.wait(0.5)

        # 3. Downward semicircular arrows

        # Downward semicircular arrows
        arrows = VGroup()

        arrow_y = -0.3

        for i in range(4):

            # Start point: midpoint between two digits
            start_x = (
                bin_mobs[i + 1].get_center()[0]
                + bin_mobs[i + 2].get_center()[0]
            ) / 2

            # End point: midpoint between the next two digits
            end_x = (
                bin_mobs[i + 2].get_center()[0]
                + bin_mobs[i + 3].get_center()[0]
            ) / 2

            arrow = CurvedArrow(
                start_point=[start_x, arrow_y, 0],
                end_point=[end_x, arrow_y, 0],
                angle=PI,
                color=YELLOW,
                stroke_width=3,
                tip_length=0.13
            )

            self.play(
                Create(arrow),
                run_time=0.45
            )

            arrows.add(arrow)


        # Shift count
        shift_text = MathTex(
            r"4\ shifts",
            font_size=30,
            color=YELLOW
        ).next_to(arrows, DOWN, buff=0.2)

        self.play(Write(shift_text))
        self.wait(1)

        # Remove arrows and shift count before normalization
        self.play(
            FadeOut(arrows),
            FadeOut(shift_text),
            run_time=0.7
        )

        self.wait(0.3)


        # The original binary expression is already visible
        current_expr = VGroup(bin_mobs, target_ellipsis)

        # Start from the second expression
        step_strings = [
            "0.0011001100",
            "0.011001100",
            "0.11001100",
            "1.1001100"
        ]

        expression_y = target_equation.get_center()[1]

        # Fade in each next expression with ellipsis
                # Keep the left position fixed
        fixed_left = target_bin.get_left()[0]
        expression_y = target_bin.get_center()[1]

        # Fade in each next expression with ellipsis
        for text in step_strings:
            next_number = MathTex(
                text,
                font_size=44
            )

            next_ellipsis = MathTex(
                r"\ldots",
                font_size=44
            ).next_to(next_number, RIGHT, buff=0.08)

            next_expr = VGroup(
                next_number,
                next_ellipsis
            )

            # Align every expression to the same left position
            next_expr.move_to([
                fixed_left + next_expr.width / 2,
                expression_y,
                0
            ])

            self.play(
                FadeOut(current_expr),
                FadeIn(next_expr),
                run_time=0.8
            )

            current_expr = next_expr
            self.wait(0.5)


        # 5. Add exponent
        multiplier = MathTex(
            r"\times 2",
            r"^{-4}",
            color=RED,
            font_size=44
        )

        multiplier.next_to(current_expr, RIGHT, buff=0.2)

        self.play(Write(multiplier))
        self.wait(1)



        # 6. Significand and exponent labels

        mantissa_brace = Brace(
            current_expr,
            DOWN,
            color=BLUE
        )

        mantissa_text = Text(
            "Significand",
            font_size=24,
            color=BLUE
        ).next_to(mantissa_brace, DOWN, buff=0.1)

        exp_brace = Brace(
            multiplier[1],
            UP,
            color=RED
        )

        exp_text = Text(
            "Exponent",
            font_size=24,
            color=RED
        ).next_to(exp_brace, UP, buff=0.1)

        self.play(
            GrowFromCenter(mantissa_brace),
            FadeIn(mantissa_text),
            GrowFromCenter(exp_brace),
            FadeIn(exp_text)
        )

        self.wait(2)

        self.play(
            FadeOut(mantissa_brace),
            FadeOut(mantissa_text),
            FadeOut(exp_brace),
            FadeOut(exp_text)
        )

        # 6. Move the normalized expression upward

        normalized_equation = VGroup(current_expr, multiplier)

        self.play(
            normalized_equation.animate.move_to(UP * 1.0),
            Unwrite(dec_label),
            run_time=1.5
        )

        self.wait(0.5)



        # 7. Create the 32-bit storage area

        cell_width = 0.16
        cell_height = 0.42
        cell_gap = 0.012

        def make_slots(count):
            return VGroup(*[
                Rectangle(
                    width=cell_width,
                    height=cell_height,
                    stroke_color=WHITE,
                    stroke_width=1.2
                )
                for _ in range(count)
            ]).arrange(RIGHT, buff=cell_gap)

        sign_slots = make_slots(1)
        exponent_slots = make_slots(8)
        fraction_slots = make_slots(23)

        bit_fields = VGroup(
            sign_slots,
            exponent_slots,
            fraction_slots
        ).arrange(RIGHT, buff=0.18)

        bit_fields.move_to(DOWN * 1.35)

        sign_label = Text(
            "Sign",
            font_size=20,
            color=YELLOW
        ).next_to(sign_slots, DOWN, buff=0.15)

        exponent_label = Text(
            "Exponent",
            font_size=20,
            color=RED
        ).next_to(exponent_slots, DOWN, buff=0.15)

        fraction_label = Text(
            "Fraction",
            font_size=20,
            color=BLUE
        ).next_to(fraction_slots, DOWN, buff=0.15)

        self.play(
            Create(bit_fields),
            FadeIn(sign_label),
            FadeIn(exponent_label),
            FadeIn(fraction_label),
            run_time=1.5
        )

        self.wait(0.5)


        # # 8. Remove the decimal expression and center the normalized number

        # self.play(
        #     Unwrite(dec_label),
        #     normalized_equation.animate.move_to(UP * 1.65),
        #     run_time=1.2
        # )

        # self.wait(0.5)

        # 9. Fill the sign bit

        # Sign bit: appear above and move into the slot
        sign_bit = MathTex(
            "0",
            font_size=30,
            color=YELLOW
        ).move_to(sign_slots[0].get_center() + UP * 0.7)

        self.play(FadeIn(sign_bit))

        self.play(
            sign_bit.animate
            .scale(22 / 30)
            .move_to(sign_slots[0].get_center()),
            run_time=0.8
        )
        self.wait(0.5)

        # 10. Calculate the biased exponent

        bias_calc = MathTex(
            r"-4 + 127 = 123",
            font_size=30,
            color=RED
        ).move_to(UP * 0.45)

        self.play(Write(bias_calc))
        self.wait(1)

        exponent_bits = "01111011"

        exponent_source = VGroup(*[
            MathTex(bit, font_size=30, color=RED)
            for bit in exponent_bits
        ]).arrange(RIGHT, buff=0.025)

        exponent_source.move_to(
            exponent_slots.get_center() + UP * 0.55
        )

        self.play(
            FadeOut(bias_calc),
            Write(exponent_source),
            run_time=0.8
        )

        self.play(
            LaggedStart(
                *[
                    bit.animate
                    .scale(20 / 30)
                    .move_to(slot.get_center())
                    for bit, slot in zip(exponent_source, exponent_slots)
                ],
                lag_ratio=0.08
            ),
            run_time=1.8
        )

        self.wait(0.5)

        # 11. Fill the fraction bits

        fraction_bits = (
            "10011001100110011001101"
        )

        fraction_source = VGroup(*[
            MathTex(bit, font_size=24, color=BLUE)
            for bit in fraction_bits
        ]).arrange(RIGHT, buff=0.025)

        fraction_source.move_to(
            fraction_slots.get_center() + UP * 0.55
        )

        self.play(
            Write(fraction_source),
            run_time=1.2
        )

        self.play(
            LaggedStart(
                *[
                    bit.animate
                    .scale(17 / 24)
                    .move_to(slot.get_center())
                    for bit, slot in zip(fraction_source, fraction_slots)
                ],
                lag_ratio=0.015
            ),
            run_time=3
        )

        self.wait(2)


Overwriting scene.py


In [3]:
#!manim -pql scene.py NormalizeBinary
!manim -pqh scene.py NormalizeBinary

Manim Community v0.22.1

STEP 1: Scene started
[11/21/26 04:32:18] INFO     Animation 0 : Using cached    ]8;id=465207;file:///usr/local/lib/python3.13/dist-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=628471;file:///usr/local/lib/python3.13/dist-packages/manim/renderer/cairo_renderer.py#105\105]8;;\
                             data (hash :                                       
                             b4c4be3af32cf95819844dc121b2d                      
                             750adc38a82f7ed092a5ce412c8c7                      
                             2c56d7)                                            
STEP 2: Title completed
[11/21/26 04:32:19] INFO     Animation 1 : Using cached    ]8;id=287565;file:///usr/local/lib/python3.13/dist-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=105595;file:///usr/local/lib/python3.13/dist-packages/manim/renderer/cairo_renderer.py#105\105]8;;\
                           

In [5]:
from IPython.display import Video, display

display(Video(
    "media/videos/scene/480p15/NormalizeBinary.mp4",
    embed=True
))